# Ejercicio 02: Model Registry con MLflow — SOLUCION

Este notebook contiene la **solucion completa** del ejercicio 02.

---

### Analogia: La vitrina de la panaderia

| Panaderia | MLflow |
|---|---|
| Cocina (donde experimentas recetas) | **Tracking** (runs, metricas, parametros) |
| Vitrina (donde pones los productos aprobados) | **Model Registry** (modelos registrados) |
| Etiqueta "Producto estrella" en la vitrina | **Alias `champion`** |
| Etiqueta "Nuevo, en evaluacion" | **Alias `candidate`** |
| Receta v1, v2, v3... | **Versiones** del modelo |

### Flujo visual:

```
Tracking (cocina)                    Registry (vitrina)              Produccion
+-----------------------+    +----------------------------+    +----------------+
| run 1: RandomForest   | -> | iris-clasificacion          | -> |                |
| run 2: otro modelo    |    |   v1 (champion)            |    |  .predict()    |
|                       |    |   v2 (candidate)           |    |                |
+-----------------------+    +----------------------------+    +----------------+
```

---

## Prerrequisitos

Tener el servidor de MLflow corriendo:

```bash
mlflow server \
  --host 127.0.0.1 \
  --port 5001 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns
```

In [ ]:
# Descomentar si necesitas instalar
# !pip install mlflow scikit-learn

---

## Parte 1: Codigo base (NO modificar, solo ejecutar)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# --- Cargar dataset Iris ---
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

print(f"Dataset: {len(X)} registros, {X.shape[1]} features")
print(f"Clases: {list(iris.target_names)}")
print(f"Distribucion: {np.bincount(y)}")

In [ ]:
# --- Dividir datos ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# --- Escalar features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features escaladas con StandardScaler")

In [ ]:
# --- Entrenar modelo ---
n_estimators = 100
max_depth = 5
random_state = 42

modelo = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
)
modelo.fit(X_train_scaled, y_train)
print("Modelo entrenado!")

In [ ]:
# --- Calcular metricas ---
y_pred = modelo.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nReporte de clasificacion:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

---

## Parte 2: Soluciones

### TODO 1: Conectar a MLflow y crear un experimento

In [ ]:
# SOLUCION TODO 1
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("iris-model-registry")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")

### TODO 2: Loggear el modelo en un run de MLflow

In [ ]:
# SOLUCION TODO 2
with mlflow.start_run(run_name="rf_iris_v1") as run:

    # 2a. Registrar parametros
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)

    # 2b. Registrar metrica
    mlflow.log_metric("accuracy", accuracy)

    # 2c. Loggear el modelo
    mlflow.sklearn.log_model(modelo, "modelo_rf")

    # 2d. Guardar run_id
    run_id = run.info.run_id


print(f"Run ID: {run_id}")
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
# Verificacion
run_data = mlflow.get_run(run_id)
print("Parametros registrados:", dict(run_data.data.params))
print("Metricas registradas: ", dict(run_data.data.metrics))
print("Artifacts:")
from mlflow.tracking import MlflowClient
client = MlflowClient()
for artifact in client.list_artifacts(run_id):
    print(f"  -> {artifact.path}")

### TODO 3: Registrar el modelo en el Model Registry

In [ ]:
# SOLUCION TODO 3

# 3a. Definir nombre del modelo
nombre_modelo = "iris-clasificacion"

# 3b. Construir model_uri
model_uri = f"runs:/{run_id}/modelo_rf"

# 3c. Registrar el modelo
mlflow.register_model(model_uri, nombre_modelo)

print(f"Modelo '{nombre_modelo}' registrado!")

### TODO 4: Asignar el alias `champion` a la version 1

In [ ]:
# SOLUCION TODO 4
client.set_registered_model_alias(nombre_modelo, "champion", "1")

print(f"Alias 'champion' asignado a version 1 de '{nombre_modelo}'")

In [ ]:
# Verificacion
modelo_info = client.get_registered_model(nombre_modelo)
print(f"Modelo: {modelo_info.name}")
print(f"Aliases: {modelo_info.aliases}")

champion = client.get_model_version_by_alias(nombre_modelo, "champion")
print(f"Champion es la version: {champion.version}")

### TODO 5: Cargar el modelo por numero de version

In [ ]:
# SOLUCION TODO 5

# 5a. Construir la URI
model_uri_v1 = f"models:/{nombre_modelo}/1"

# 5b. Cargar el modelo
modelo_cargado = mlflow.sklearn.load_model(model_uri_v1)

print(f"Modelo cargado: {type(modelo_cargado).__name__}")
print(f"Desde: {model_uri_v1}")

### TODO 6: Cargar el modelo por alias (forma recomendada para produccion)

In [ ]:
# SOLUCION TODO 6

# 6a. Obtener la version del champion
champion_info = client.get_model_version_by_alias(nombre_modelo, "champion")

# 6b. Construir la URI
model_uri_champion = f"models:/{nombre_modelo}/{champion_info.version}"

# 6c. Cargar el modelo
modelo_champion = mlflow.sklearn.load_model(model_uri_champion)

print(f"Modelo champion cargado (version {champion_info.version})")
print(f"Desde: {model_uri_champion}")

### TODO 7: Hacer predicciones con el modelo cargado

In [ ]:
# SOLUCION TODO 7

# 7a. Predecir
y_pred_registry = modelo_champion.predict(X_test_scaled)

# 7b. Calcular accuracy
accuracy_registry = accuracy_score(y_test, y_pred_registry)

# 7c. Comparar
print(f"Accuracy original:  {accuracy:.4f}")
print(f"Accuracy Registry:  {accuracy_registry:.4f}")
print(f"Son iguales: {accuracy == accuracy_registry}")

---

## Parte 3: Registrar una segunda version (v2)

### TODO 8: Entrenar un segundo modelo con hiperparametros diferentes

In [ ]:
# SOLUCION TODO 8
with mlflow.start_run(run_name="rf_iris_v2") as run2:

    # 8a. Nuevos hiperparametros
    n_estimators_v2 = 200
    max_depth_v2 = 10

    # 8b. Registrar parametros
    mlflow.log_param("n_estimators", n_estimators_v2)
    mlflow.log_param("max_depth", max_depth_v2)
    mlflow.log_param("random_state", random_state)

    # 8c. Crear y entrenar modelo
    modelo_v2 = RandomForestClassifier(
        n_estimators=n_estimators_v2,
        max_depth=max_depth_v2,
        random_state=random_state,
    )
    modelo_v2.fit(X_train_scaled, y_train)

    # 8d. Predecir y calcular accuracy
    y_pred_v2 = modelo_v2.predict(X_test_scaled)
    accuracy_v2 = accuracy_score(y_test, y_pred_v2)

    # 8e. Registrar metrica
    mlflow.log_metric("accuracy", accuracy_v2)

    # 8f. Loggear Y registrar modelo en un solo paso
    mlflow.sklearn.log_model(
        modelo_v2,
        "modelo_rf",
        registered_model_name=nombre_modelo,
    )

    # 8g. Guardar run_id
    run_id_v2 = run2.info.run_id


print(f"Run ID v2: {run_id_v2}")
print(f"Accuracy v2: {accuracy_v2:.4f}")

### TODO 9: Asignar alias `candidate` a la version 2

In [ ]:
# SOLUCION TODO 9
client.set_registered_model_alias(nombre_modelo, "candidate", "2")

print("Alias 'candidate' asignado a version 2")

### Verificacion final

In [ ]:
# Verificacion: mostrar estado completo del Registry
modelo_info = client.get_registered_model(nombre_modelo)
print(f"Modelo: {modelo_info.name}")
print(f"Aliases: {modelo_info.aliases}")
print()

# Mostrar todas las versiones
for alias_name in ["champion", "candidate"]:
    version_info = client.get_model_version_by_alias(nombre_modelo, alias_name)
    run_data = mlflow.get_run(version_info.run_id)
    acc = run_data.data.metrics.get("accuracy", "N/A")
    print(f"  {alias_name:10s} -> Version {version_info.version} | Accuracy: {acc}")

### Bonus: Promover candidate a champion

In [ ]:
# Bonus: Promover candidate a champion
client.set_registered_model_alias(nombre_modelo, "champion", "2")
print("Champion actualizado a version 2!")
print(f"Aliases: {client.get_registered_model(nombre_modelo).aliases}")

---

## Resumen

| Concepto | Funcion | Para que sirve |
|---|---|---|
| Loggear modelo | `mlflow.sklearn.log_model()` | Guardar modelo como artifact dentro de un run |
| Registrar modelo | `mlflow.register_model()` | Crear nombre + version en el Registry |
| Loggear + Registrar | `log_model(..., registered_model_name=)` | Hacer ambas cosas en un paso |
| Asignar alias | `client.set_registered_model_alias()` | Etiquetar version como champion/candidate |
| Cargar por version | `models:/nombre/1` | Obtener una version especifica |
| Cargar por alias | `client.get_model_version_by_alias()` | Obtener la version champion |

### Diagrama del flujo completo:

```
Entrenar           Loggear              Registrar            Etiquetar          Cargar
modelo.fit() -->  log_model()  -->  register_model()  -->  set_alias()  -->  load_model()
  (sklearn)        (tracking)          (registry)        (champion/           (produccion)
                                                          candidate)
```